# Vígil.ia — Voto temporal EXIGENTE (yolo11s + yolo11n)

Ataca as alucinações observadas no vídeo (11s → `broken`, 11n → `immature`) **sem
retreinar**: muda a regra de decisão por grão.

**Regra atual (maioria simples):** classe mais votada ganha, mesmo com 40% dos votos.
**Regra exigente:** *defeito precisa provar; intacto é o benefício da dúvida.*
Um grão só é marcado como defeito se a classe de defeito tiver **≥ RATIO dos votos**
(ponderados por confiança). Senão, cai para `intact`.

O notebook faz **1 passada de tracking por modelo** (guarda os votos e as caixas) e
depois aplica **várias regras offline** — barato de experimentar RATIOs diferentes.

Mede em 2 lugares:
1. **Vídeos só-intacto** (verdade conhecida): quanto a regra reduz o falso-defeito.
2. **`teste_soja` (misto)**: gera vídeo anotado lado a lado (simples vs exigente).

> ⚠️ Honestidade: essa regra **troca** falso-defeito por risco de deixar defeito
> passar. Nos vídeos só-intacto ela SÓ melhora (por construção). O outro lado —
> quanto defeito real ela mascara — só é medível com os vídeos de defeito.

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Caminhos + configuração

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# modelos leves — usa o base12k se existir, senão o det_v3 (11s) / variantes do 11n
def first_existing(*paths):
    return next((p for p in paths if os.path.exists(p)), None)

MODELS = {}
p = first_existing('/content/drive/MyDrive/soja_yolo11s_base12k.pt',
                   '/content/drive/MyDrive/soja_yolo11s_det_v3.pt')
if p: MODELS['yolo11s'] = p
p = first_existing('/content/drive/MyDrive/soja_yolo11n_base12k.pt',
                   '/content/drive/MyDrive/soja_yolo11n_baseline.pt')
if p: MODELS['yolo11n'] = p
assert MODELS, 'Nenhum modelo 11s/11n encontrado no Drive!'
print('modelos:', MODELS)

# vídeos só-intacto (verdade conhecida) — p/ medir a redução de falso-defeito
INTACT_DIR = '/content/drive/MyDrive/Vídeos para treino/Treino/Intacto'
assert os.path.isdir(INTACT_DIR), f'pasta não encontrada: {INTACT_DIR}'

# vídeo misto — p/ os vídeos anotados
VIDEO_TESTE = first_existing('/content/drive/MyDrive/teste_soja.mp4',
                             '/content/drive/MyDrive/teste_soja.avi')
assert VIDEO_TESTE, 'teste_soja.(mp4|avi) não encontrado!'

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
DEFECTS = [n for n in NAMES if n != 'intact']
CONF, IMGSZ, IOU = 0.35, 640, 0.5
RATIOS = [0.50, 0.60, 0.70, 0.80]   # exigências testadas (0.50 ~ maioria simples)
print('vídeo teste:', VIDEO_TESTE)

## 2. Coletor de votos — 1 passada de tracking por (modelo, vídeo)
Guarda `votes[tid][classe] += conf` e as caixas por frame. As regras são aplicadas
depois, offline — rode uma vez e experimente quantos RATIOs quiser.

In [ ]:
from collections import defaultdict, Counter
import glob, cv2
from ultralytics import YOLO

def coletar(model, src):
    """-> votes: tid -> Counter(classe: soma de conf); dets: frame_idx -> [(tid, x1,y1,x2,y2)]"""
    votes, dets = defaultdict(Counter), defaultdict(list)
    for k, r in enumerate(model.track(source=src, imgsz=IMGSZ, iou=IOU, conf=CONF,
                                      agnostic_nms=True, tracker='bytetrack.yaml',
                                      stream=True, persist=True, verbose=False)):
        if r.boxes.id is None:
            continue
        for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                                    r.boxes.id.int().tolist(),
                                    r.boxes.cls.int().tolist(),
                                    r.boxes.conf.tolist()):
            votes[tid][NAMES[c]] += cf
            dets[k].append((tid, *xyxy.tolist()))
    return votes, dets

def veredito_simples(cnt):
    return cnt.most_common(1)[0][0]

def veredito_exigente(cnt, ratio):
    """Defeito só com >= ratio dos votos ponderados; senão intact (benefício da dúvida)."""
    top, top_w = cnt.most_common(1)[0]
    if top == 'intact':
        return 'intact'
    return top if top_w >= ratio * sum(cnt.values()) else 'intact'

print('funções prontas')

## 3. Vídeos só-intacto — quanto cada exigência reduz o falso-defeito
Verdade conhecida: todo grão é `intact`. Compara maioria simples vs RATIOs.

In [ ]:
vids_intact = sorted(v for v in glob.glob(f'{INTACT_DIR}/*')
                     if v.lower().endswith(('.mp4', '.mov', '.avi', '.mkv')))
assert vids_intact, f'nenhum vídeo em {INTACT_DIR}'
print(f'{len(vids_intact)} vídeo(s) só-intacto\n')

intact_votes = {}   # tag -> lista de Counters (1 por grão), p/ reuso na análise
for tag, pt in MODELS.items():
    model = YOLO(pt)
    all_votes = []
    for vid in vids_intact:
        v, _ = coletar(model, vid)
        all_votes += list(v.values())
    intact_votes[tag] = all_votes

    n = len(all_votes)
    print(f'===== {tag} ({n} grãos, verdade = 100% intact) =====')
    base = sum(1 for cnt in all_votes if veredito_simples(cnt) == 'intact')
    print(f'  maioria simples : {base}/{n} intact ({100*base/n:.1f}%)  '
          f'falso-defeito {100*(n-base)/n:.1f}%')
    for ratio in RATIOS:
        ok = sum(1 for cnt in all_votes if veredito_exigente(cnt, ratio) == 'intact')
        errs = Counter(veredito_exigente(cnt, ratio) for cnt in all_votes
                       if veredito_exigente(cnt, ratio) != 'intact')
        print(f'  exigente {ratio:.2f}   : {ok}/{n} intact ({100*ok/n:.1f}%)  '
          f'falso-defeito {100*(n-ok)/n:.1f}%  {dict(errs) if errs else ""}')
    print()

print('Escolha o RATIO no cotovelo: onde o falso-defeito cai bastante sem precisar')
print('de exigência absurda (0.80+ mascara defeito real demais).')

## 4. `teste_soja` (misto) — vídeo anotado: simples vs exigente
Para cada modelo, re-renderiza o vídeo com as DUAS regras usando a mesma passada.
Cinza = intact, colorido = defeito. Compare os pares no Drive.

In [ ]:
RATIO_ESCOLHIDO = 0.70   # <- ajuste com base no cotovelo da célula anterior

COLORS = {'intact': (140, 140, 140), 'immature': (60, 200, 200), 'broken': (200, 100, 160),
          'skin-damaged': (60, 160, 255), 'spotted': (80, 80, 230)}

def render(src, dets, verdicts, out_path):
    cap = cv2.VideoCapture(src)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    writer, k = None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        for tid, x1, y1, x2, y2 in dets.get(k, []):
            cls = verdicts.get(tid, 'intact')
            color = COLORS[cls]
            thick = 2 if cls == 'intact' else 3
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick)
            cv2.putText(frame, f'#{tid} {cls}', (x1, max(18, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        writer.write(frame)
        k += 1
    cap.release(); writer.release()

for tag, pt in MODELS.items():
    print(f'>>> {tag}: coletando votos no teste_soja…')
    votes, dets = coletar(YOLO(pt), VIDEO_TESTE)

    for regra, fn in [('simples', veredito_simples),
                      (f'exigente{int(RATIO_ESCOLHIDO*100)}',
                       lambda cnt: veredito_exigente(cnt, RATIO_ESCOLHIDO))]:
        verdicts = {tid: fn(cnt) for tid, cnt in votes.items()}
        dist = Counter(verdicts.values())
        out = f'/content/voto_{tag}_{regra}.mp4'
        render(VIDEO_TESTE, dets, verdicts, out)
        !cp {out} /content/drive/MyDrive/
        n = len(verdicts)
        print(f'  {regra:12s}: {n} grãos | intact {dist.get("intact",0)} '
              f'({100*dist.get("intact",0)/max(n,1):.0f}%) | {dict(dist)}')
        print(f'  {"":12s}  -> Drive/voto_{tag}_{regra}.mp4')
    print()

print('Assista os pares (simples vs exigente) e veja se as alucinações')
print('(11s->broken, 11n->immature) sumiram sem apagar defeito óbvio.')

## Como fechar
1. Célula §3: escolha o RATIO no cotovelo (chute inicial: 0.70).
2. Célula §4: assista os pares no Drive — a alucinação sumiu? defeito óbvio continuou marcado?
3. O RATIO vencedor vai pro pipeline final (demo/app) na função `veredito_exigente`.
4. **Pendência que não some:** medir quanto defeito real o filtro mascara exige os
   vídeos de `broken`/`immature`/`spotted`/`skin-damaged`. Quando existirem, rode a
   §3 apontando pra essas pastas — a métrica vira "defeito que sobreviveu ao filtro".